# 📊 Unemployment in India — Exploratory Data Analysis
### CodeAlpha Data Science Internship | Task 2

---

**Objective:** Analyze unemployment rate data across Indian states (May 2019 – Jun 2020), investigate the COVID-19 impact, identify seasonal trends, and derive policy insights.

| Field | Detail |
|-------|--------|
| 📅 Period | May 2019 – June 2020 |
| 🌍 Coverage | 28 Indian States / UTs |
| 📦 Source | CMIE (Centre for Monitoring Indian Economy) |
| 🔢 Records | 768 raw → 740 after cleaning |

---

## 📋 Table of Contents
1. [Imports & Setup](#1-imports--setup)
2. [Data Loading & Cleaning](#2-data-loading--cleaning)
3. [Exploratory Data Analysis](#3-exploratory-data-analysis)
4. [National Trend Analysis](#4-national-trend-analysis)
5. [Regional Analysis](#5-regional-analysis)
6. [COVID-19 Impact](#6-covid-19-impact)
7. [Rural vs Urban Analysis](#7-rural-vs-urban-analysis)
8. [Seasonal Pattern Analysis](#8-seasonal-pattern-analysis)
9. [Correlation & Statistical Analysis](#9-correlation--statistical-analysis)
10. [Regional Heatmap](#10-regional-heatmap)
11. [Policy Insights Summary](#11-policy-insights-summary)


## 1. Imports & Setup

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
warnings.filterwarnings("ignore")
os.makedirs("images", exist_ok=True)

# ── Colour palette ─────────────────────────────────────────
BG, CARD   = "#0d0f18", "#141726"
TEAL       = "#2ee8b5"
ORANGE     = "#ff7043"
YELLOW     = "#ffc947"
PURPLE     = "#b39ddb"
BLUE       = "#64b5f6"
PINK       = "#f48fb1"
MUTED      = "#5c6880"
TEXT       = "#e8edf8"
WHITE      = "#ffffff"

plt.rcParams.update({
    "figure.facecolor":  BG,   "axes.facecolor":  "#141726",
    "text.color":        TEXT, "axes.labelcolor": TEXT,
    "xtick.color":       MUTED,"ytick.color":     MUTED,
    "axes.edgecolor":    "#252840", "grid.color":  "#1e2236",
    "axes.spines.top":   False,"axes.spines.right":False,
    "font.family":       "DejaVu Sans", "font.size": 9,
})
pct  = mticker.FuncFormatter(lambda x,_: f"{x:.0f}%")
pct1 = mticker.FuncFormatter(lambda x,_: f"{x:.1f}%")

print("✅ Libraries loaded. Palette set.")


## 2. Data Loading & Cleaning

In [ ]:
# ── Load raw CSV ───────────────────────────────────────────
df_raw = pd.read_csv("Unemployment in India.csv")
df_raw.columns = [c.strip() for c in df_raw.columns]

print("📋 RAW DATA SNAPSHOT")
print(f"   Shape   : {df_raw.shape}")
print(f"   Columns : {list(df_raw.columns)}")
df_raw.head(6)


In [ ]:
# ── Missing value analysis ─────────────────────────────────
print("🔍 MISSING VALUES BEFORE CLEANING")
print(df_raw.isnull().sum().to_string())


In [ ]:
# ── Clean ──────────────────────────────────────────────────
df = df_raw.dropna(subset=["Region"]).copy()
df.columns = ["Region","Date","Frequency",
              "Unemployment_Rate","Estimated_Employed",
              "Labour_Participation_Rate","Area"]
df["Date"]       = pd.to_datetime(df["Date"].str.strip(), dayfirst=True)
df["Month"]      = df["Date"].dt.month
df["Month_Name"] = df["Date"].dt.strftime("%b")
df["Year"]       = df["Date"].dt.year
df["COVID"]      = df["Date"] >= "2020-03-01"

# Save cleaned file
df.to_csv("Cleaned_Data.csv", index=False)

print("✅ CLEANED DATASET")
print(f"   Shape      : {df.shape}")
print(f"   Date range : {df['Date'].min().strftime('%b %Y')} → {df['Date'].max().strftime('%b %Y')}")
print(f"   Regions    : {df['Region'].nunique()} states/UTs")
print(f"   Saved      : Cleaned_Data.csv")
df.info()


In [ ]:
# ── Descriptive statistics ─────────────────────────────────
print("📊 DESCRIPTIVE STATISTICS")
df[["Unemployment_Rate","Estimated_Employed","Labour_Participation_Rate"]].describe().round(2)


## 3. Exploratory Data Analysis

In [ ]:
# ── Compute key metrics ────────────────────────────────────
pre_covid   = df.loc[~df["COVID"], "Unemployment_Rate"].mean()
covid_avg   = df.loc[ df["COVID"], "Unemployment_Rate"].mean()
rural_avg   = df.loc[df["Area"]=="Rural", "Unemployment_Rate"].mean()
urban_avg   = df.loc[df["Area"]=="Urban", "Unemployment_Rate"].mean()
region_avg  = df.groupby("Region")["Unemployment_Rate"].mean().sort_values(ascending=False)
monthly     = df.groupby("Date")["Unemployment_Rate"].mean().reset_index()

print("="*55)
print("  KEY STATISTICS — UNEMPLOYMENT IN INDIA")
print("="*55)
print(f"  Overall avg        : {df['Unemployment_Rate'].mean():.2f}%")
print(f"  Pre-COVID avg      : {pre_covid:.2f}%")
print(f"  COVID-period avg   : {covid_avg:.2f}%")
print(f"  COVID shock        : +{covid_avg-pre_covid:.2f} pp ({((covid_avg/pre_covid)-1)*100:.0f}% surge)")
peak = df.loc[df['Unemployment_Rate'].idxmax()]
print(f"  Peak reading       : {peak['Unemployment_Rate']:.1f}% — {peak['Region']} ({peak['Date'].strftime('%b %Y')})")
print(f"  Rural avg          : {rural_avg:.2f}%")
print(f"  Urban avg          : {urban_avg:.2f}%")
print()
print("  TOP 5 HIGHEST STATES:")
for r,v in region_avg.head(5).items():
    print(f"    {r:25s}: {v:.2f}%")
print()
print("  TOP 5 LOWEST STATES:")
for r,v in region_avg.tail(5).items():
    print(f"    {r:25s}: {v:.2f}%")


## 4. National Trend Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(16, 5), facecolor=BG)
ax.set_facecolor("#141726")

covid_s = pd.Timestamp("2020-03-01")
covid_e = pd.Timestamp("2020-06-30")
ax.axvspan(covid_s, covid_e, color=ORANGE, alpha=0.11, label="COVID-19 Lockdown")
ax.axhline(pre_covid, color=YELLOW, linestyle="--", lw=1.5, alpha=0.8,
           label=f"Pre-COVID avg: {pre_covid:.1f}%")
ax.plot(monthly["Date"], monthly["Unemployment_Rate"],
        color=TEAL, lw=2.8, zorder=4)
ax.fill_between(monthly["Date"], monthly["Unemployment_Rate"],
                alpha=0.15, color=TEAL)

peak_date = monthly.loc[monthly["Unemployment_Rate"].idxmax(), "Date"]
peak_v    = monthly["Unemployment_Rate"].max()
ax.annotate(f"Lockdown Peak\n{peak_v:.1f}%",
            xy=(peak_date, peak_v),
            xytext=(pd.Timestamp("2020-05-20"), peak_v-5.5),
            arrowprops=dict(arrowstyle="->", color=ORANGE, lw=1.6),
            color=ORANGE, fontsize=11, fontweight="bold")

for _, row in monthly.iterrows():
    ax.scatter(row["Date"], row["Unemployment_Rate"], color=TEAL, s=32, zorder=5)
    ax.text(row["Date"], row["Unemployment_Rate"]+0.8,
            f"{row['Unemployment_Rate']:.1f}", ha="center", fontsize=8, color=TEAL)

ax.set_title("📈  National Average Unemployment Rate — Monthly Trend  (May 2019 – Jun 2020)",
             color=WHITE, fontsize=14, fontweight="bold", pad=12, loc="left")
ax.set_ylabel("Unemployment Rate (%)", fontsize=10)
ax.yaxis.set_major_formatter(pct)
ax.legend(fontsize=10, framealpha=0)
ax.grid(True, axis="y", alpha=0.5)
plt.tight_layout()
plt.savefig("images/01_national_trend.png", dpi=150, bbox_inches="tight", facecolor=BG)
plt.show()
print("Saved → images/01_national_trend.png")


In [ ]:
# ── Labour Participation Rate trend ───────────────────────
lpr = df.groupby("Date")["Labour_Participation_Rate"].mean().reset_index()

fig, ax = plt.subplots(figsize=(16, 4), facecolor=BG)
ax.set_facecolor("#141726")
ax.axvspan(covid_s, covid_e, color=ORANGE, alpha=0.10, label="COVID-19 Lockdown")
ax.plot(lpr["Date"], lpr["Labour_Participation_Rate"],
        color=PINK, lw=2.5, marker="o", ms=6)
ax.fill_between(lpr["Date"], lpr["Labour_Participation_Rate"],
                alpha=0.12, color=PINK)
for _, row in lpr.iterrows():
    ax.text(row["Date"], row["Labour_Participation_Rate"]+0.25,
            f"{row['Labour_Participation_Rate']:.1f}", ha="center", fontsize=8, color=PINK)
ax.set_title("👥  Labour Force Participation Rate — Monthly",
             color=WHITE, fontsize=13, fontweight="bold", pad=10, loc="left")
ax.set_ylabel("Participation Rate (%)", fontsize=10)
ax.yaxis.set_major_formatter(pct)
ax.legend(fontsize=9, framealpha=0)
ax.grid(True, axis="y", alpha=0.4)
plt.tight_layout()
plt.savefig("images/02_labour_participation.png", dpi=150, bbox_inches="tight", facecolor=BG)
plt.show()


## 5. Regional Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7), facecolor=BG)

# Top 10
ax = axes[0]
ax.set_facecolor("#141726")
top10 = region_avg.head(10)
bcols = [ORANGE if v>20 else (YELLOW if v>14 else TEAL) for v in top10.values]
bars = ax.barh(top10.index[::-1], top10.values[::-1],
               color=bcols[::-1], height=0.65, alpha=0.88)
ax.axvline(pre_covid, color=YELLOW, linestyle="--", lw=1.3, alpha=0.7,
           label=f"National avg {pre_covid:.1f}%")
for bar, val in zip(bars, top10.values[::-1]):
    ax.text(val+0.3, bar.get_y()+bar.get_height()/2,
            f"{val:.1f}%", va="center", fontsize=9, color=TEXT)
ax.set_title("🔴  Top 10 — Highest Unemployment States",
             color=WHITE, fontsize=12, fontweight="bold", pad=10, loc="left")
ax.set_xlabel("Avg Unemployment Rate (%)", fontsize=10)
ax.xaxis.set_major_formatter(pct)
ax.legend(fontsize=9, framealpha=0)
ax.grid(True, axis="x", alpha=0.4)

# Bottom 10
ax = axes[1]
ax.set_facecolor("#141726")
bot10 = region_avg.tail(10).sort_values()
bars3 = ax.barh(bot10.index, bot10.values, color=BLUE, height=0.65, alpha=0.88)
ax.axvline(pre_covid, color=YELLOW, linestyle="--", lw=1.3, alpha=0.7,
           label=f"National avg {pre_covid:.1f}%")
for bar, val in zip(bars3, bot10.values):
    ax.text(val+0.2, bar.get_y()+bar.get_height()/2,
            f"{val:.1f}%", va="center", fontsize=9, color=TEXT)
ax.set_title("🟢  Top 10 — Lowest Unemployment States",
             color=WHITE, fontsize=12, fontweight="bold", pad=10, loc="left")
ax.set_xlabel("Avg Unemployment Rate (%)", fontsize=10)
ax.xaxis.set_major_formatter(pct)
ax.legend(fontsize=9, framealpha=0)
ax.grid(True, axis="x", alpha=0.4)

plt.suptitle("Regional Unemployment Ranking — All 28 States/UTs",
             color=WHITE, fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("images/03_regional_ranking.png", dpi=150, bbox_inches="tight", facecolor=BG)
plt.show()


## 6. COVID-19 Impact

In [ ]:
pre_r  = df[~df["COVID"]].groupby("Region")["Unemployment_Rate"].mean()
cov_r  = df[ df["COVID"]].groupby("Region")["Unemployment_Rate"].mean()
impact = (cov_r - pre_r).dropna().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(18, 7), facecolor=BG)

# COVID shock bar
ax = axes[0]
ax.set_facecolor("#141726")
top12 = impact.head(12)
bcols = [ORANGE if v>15 else (YELLOW if v>8 else PINK) for v in top12.values]
ax.barh(top12.index[::-1], top12.values[::-1], color=bcols[::-1],
        height=0.65, alpha=0.88)
for i,(r,v) in enumerate(zip(top12.index[::-1], top12.values[::-1])):
    ax.text(v+0.3, i, f"+{v:.1f}pp", va="center", fontsize=9, color=TEXT)
ax.set_title("🦠  COVID-19 Shock — Worst-Hit States",
             color=WHITE, fontsize=12, fontweight="bold", pad=10, loc="left")
ax.set_xlabel("Change in Unemployment (percentage points)", fontsize=10)
ax.grid(True, axis="x", alpha=0.4)

# Pre vs During comparison
ax = axes[1]
ax.set_facecolor("#141726")
compare = pd.DataFrame({"Pre-COVID": pre_r, "During COVID": cov_r}).dropna()
x = range(len(compare))
w = 0.38
ax.bar([i-w/2 for i in x], compare["Pre-COVID"].values,
       width=w, color=TEAL, alpha=0.85, label="Pre-COVID")
ax.bar([i+w/2 for i in x], compare["During COVID"].values,
       width=w, color=ORANGE, alpha=0.85, label="During COVID")
ax.set_xticks(list(x))
ax.set_xticklabels(compare.index, rotation=45, ha="right", fontsize=7.5)
ax.set_title("📊  Pre vs During COVID — All States",
             color=WHITE, fontsize=12, fontweight="bold", pad=10, loc="left")
ax.set_ylabel("Unemployment Rate (%)", fontsize=10)
ax.yaxis.set_major_formatter(pct)
ax.legend(fontsize=9, framealpha=0)
ax.grid(True, axis="y", alpha=0.4)

plt.suptitle(f"COVID-19 Impact  ·  Shock: +{covid_avg-pre_covid:.1f}pp  ({((covid_avg/pre_covid)-1)*100:.0f}% surge)",
             color=WHITE, fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("images/04_covid_impact.png", dpi=150, bbox_inches="tight", facecolor=BG)
plt.show()
print(f"COVID shock magnitude: Pre-COVID {pre_covid:.2f}% → COVID-period {covid_avg:.2f}% (+{covid_avg-pre_covid:.2f}pp)")


## 7. Rural vs Urban Analysis

In [ ]:
rural_m = df[df["Area"]=="Rural"].groupby("Date")["Unemployment_Rate"].mean()
urban_m = df[df["Area"]=="Urban"].groupby("Date")["Unemployment_Rate"].mean()

fig, axes = plt.subplots(1, 2, figsize=(18, 5), facecolor=BG)

# Time series
ax = axes[0]
ax.set_facecolor("#141726")
ax.axvspan(covid_s, covid_e, color=ORANGE, alpha=0.09)
ax.plot(rural_m.index, rural_m.values, color=TEAL,   lw=2.5,
        marker="o", ms=5, label=f"Rural (avg {rural_avg:.1f}%)")
ax.plot(urban_m.index, urban_m.values, color=PURPLE, lw=2.5,
        marker="s", ms=5, label=f"Urban (avg {urban_avg:.1f}%)")
ax.fill_between(rural_m.index, rural_m.values, alpha=0.10, color=TEAL)
ax.fill_between(urban_m.index, urban_m.values, alpha=0.10, color=PURPLE)
ax.set_title("🌾  Rural vs Urban — Unemployment Over Time",
             color=WHITE, fontsize=12, fontweight="bold", pad=10, loc="left")
ax.set_ylabel("Rate (%)", fontsize=10)
ax.yaxis.set_major_formatter(pct)
ax.legend(fontsize=10, framealpha=0)
ax.grid(True, axis="y", alpha=0.4)

# Gap chart
ax = axes[1]
ax.set_facecolor("#141726")
ru_state = df.groupby(["Region","Area"])["Unemployment_Rate"].mean().unstack()
gap = (ru_state.get("Urban", pd.Series()) - ru_state.get("Rural", pd.Series())).dropna().sort_values(ascending=False)
ax.bar(gap.index, gap.values,
       color=[ORANGE if v>0 else TEAL for v in gap.values], alpha=0.85)
ax.axhline(0, color=MUTED, lw=0.8)
ax.set_title("🏙️  Urban − Rural Gap by State (pp)",
             color=WHITE, fontsize=12, fontweight="bold", pad=10, loc="left")
ax.set_ylabel("Urban − Rural (pp)", fontsize=10)
ax.set_xticklabels(gap.index, rotation=45, ha="right", fontsize=7.5)
ax.grid(True, axis="y", alpha=0.4)

plt.tight_layout()
plt.savefig("images/05_rural_urban.png", dpi=150, bbox_inches="tight", facecolor=BG)
plt.show()
print(f"Urban avg: {urban_avg:.2f}%  |  Rural avg: {rural_avg:.2f}%  |  Gap: {urban_avg-rural_avg:.2f}pp")


## 8. Seasonal Pattern Analysis

In [ ]:
month_order  = [5,6,7,8,9,10,11,12,1,2,3,4]
month_labels = ["May","Jun","Jul","Aug","Sep","Oct","Nov","Dec","Jan","Feb","Mar","Apr"]
seasonal = df.groupby("Month")["Unemployment_Rate"].mean()
vals = [seasonal.get(m, 0) for m in month_order]
bcols = [ORANGE if m in [3,4] else (YELLOW if m in [5,6] else BLUE) for m in month_order]

fig, axes = plt.subplots(1, 2, figsize=(18, 5), facecolor=BG)

# Seasonal bar
ax = axes[0]
ax.set_facecolor("#141726")
bars = ax.bar(month_labels, vals, color=bcols, width=0.72, alpha=0.88)
ax.axhline(df["Unemployment_Rate"].mean(), color=YELLOW, linestyle="--", lw=1.3,
           label=f"Overall avg {df['Unemployment_Rate'].mean():.1f}%", alpha=0.8)
for bar, val in zip(bars, vals):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.2,
            f"{val:.1f}", ha="center", fontsize=8.5, color=TEXT)
ax.set_title("📅  Seasonal Pattern — Avg Unemployment by Month",
             color=WHITE, fontsize=12, fontweight="bold", pad=10, loc="left")
ax.set_ylabel("Avg Rate (%)", fontsize=10)
ax.yaxis.set_major_formatter(pct1)
ax.legend(handles=[
    mpatches.Patch(color=YELLOW,  label=f"Overall avg {df['Unemployment_Rate'].mean():.1f}%"),
    mpatches.Patch(color=ORANGE,  label="COVID period (Mar–Apr)"),
    mpatches.Patch(color=BLUE,    label="Normal months"),
], fontsize=9, framealpha=0)
ax.grid(True, axis="y", alpha=0.4)

# Box plot
ax = axes[1]
ax.set_facecolor("#141726")
data_box = [df[df["Month"]==m]["Unemployment_Rate"].values for m in month_order]
bp = ax.boxplot(data_box, labels=month_labels, patch_artist=True,
                medianprops=dict(color=WHITE, lw=1.5),
                whiskerprops=dict(color=MUTED),
                capprops=dict(color=MUTED),
                flierprops=dict(marker="o", markerfacecolor=ORANGE,
                                markersize=3, alpha=0.5))
for patch, col in zip(bp["boxes"], bcols):
    patch.set_facecolor(col); patch.set_alpha(0.55)
ax.set_title("📦  Monthly Box Plot — Spread Across States",
             color=WHITE, fontsize=12, fontweight="bold", pad=10, loc="left")
ax.set_ylabel("Unemployment Rate (%)", fontsize=10)
ax.yaxis.set_major_formatter(pct)
ax.grid(True, axis="y", alpha=0.4)

plt.tight_layout()
plt.savefig("images/06_seasonal.png", dpi=150, bbox_inches="tight", facecolor=BG)
plt.show()


## 9. Correlation & Statistical Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5), facecolor=BG)

# Correlation heatmap
ax = axes[0]
ax.set_facecolor("#141726")
corr = df[["Unemployment_Rate","Estimated_Employed","Labour_Participation_Rate"]].corr()
cmap_corr = LinearSegmentedColormap.from_list("corr",
    ["#ff7043","#141726","#2ee8b5"])
im = ax.imshow(corr.values, cmap=cmap_corr, vmin=-1, vmax=1, aspect="auto")
ax.set_xticks(range(3))
ax.set_yticks(range(3))
labels = ["Unemployment\nRate", "Estimated\nEmployed", "Labour\nParticipation"]
ax.set_xticklabels(labels, fontsize=9, color=TEXT)
ax.set_yticklabels(labels, fontsize=9, color=TEXT)
for i in range(3):
    for j in range(3):
        ax.text(j, i, f"{corr.values[i,j]:.2f}",
                ha="center", va="center", fontsize=12,
                fontweight="bold", color=WHITE)
plt.colorbar(im, ax=ax, fraction=0.04, pad=0.02).ax.yaxis.set_tick_params(color=MUTED)
ax.set_title("🔗  Correlation Matrix", color=WHITE,
             fontsize=12, fontweight="bold", pad=10, loc="left")

# Scatter: Unemployment vs Labour Participation
ax = axes[1]
ax.set_facecolor("#141726")
colors_sc = [ORANGE if c else TEAL for c in df["COVID"]]
ax.scatter(df["Labour_Participation_Rate"], df["Unemployment_Rate"],
           c=colors_sc, alpha=0.45, s=18, edgecolors="none")
from scipy.stats import linregress
sl, ic, r, p, _ = linregress(df["Labour_Participation_Rate"], df["Unemployment_Rate"])
xs = np.linspace(df["Labour_Participation_Rate"].min(),
                 df["Labour_Participation_Rate"].max(), 100)
ax.plot(xs, sl*xs+ic, color=YELLOW, lw=2, linestyle="--",
        label=f"Trend  (r={r:.2f})")
ax.set_title("📉  Labour Participation vs Unemployment Rate",
             color=WHITE, fontsize=12, fontweight="bold", pad=10, loc="left")
ax.set_xlabel("Labour Participation Rate (%)", fontsize=10)
ax.set_ylabel("Unemployment Rate (%)", fontsize=10)
ax.legend(handles=[
    mpatches.Patch(color=TEAL,   label="Pre-COVID"),
    mpatches.Patch(color=ORANGE, label="COVID Period"),
    mpatches.Patch(color=YELLOW, label=f"Trend (r={r:.2f})"),
], fontsize=9, framealpha=0)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("images/07_correlation.png", dpi=150, bbox_inches="tight", facecolor=BG)
plt.show()
print(f"Correlation (Labour Participation vs Unemployment): r = {r:.3f}  (p = {p:.4f})")


## 10. Regional Heatmap

In [ ]:
pivot = df.pivot_table(index="Region", columns="Date",
                      values="Unemployment_Rate", aggfunc="mean")
pivot.columns = pivot.columns.strftime("%b\n%Y")

fig, ax = plt.subplots(figsize=(20, 11), facecolor=BG)
ax.set_facecolor(BG)
cmap_h = LinearSegmentedColormap.from_list("india",
    ["#0d2b3e","#0b6e4f","#2ee8b5","#ffc947","#ff7043","#c0392b"])
import matplotlib
norm = matplotlib.colors.Normalize(vmin=0, vmax=pivot.values.max())
im   = ax.imshow(pivot.values, aspect="auto", cmap=cmap_h, norm=norm)

ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns, fontsize=8, color=MUTED)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index, fontsize=9, color=TEXT)

for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        v = pivot.values[i,j]
        if not np.isnan(v):
            ax.text(j, i, f"{v:.0f}", ha="center", va="center",
                    fontsize=6.5,
                    color=WHITE if v>20 else "#0d0f18",
                    fontweight="bold" if v>30 else "normal")

cb = plt.colorbar(im, ax=ax, fraction=0.015, pad=0.01)
cb.set_label("Unemployment Rate (%)", color=TEXT, fontsize=10)
plt.setp(cb.ax.yaxis.get_ticklabels(), color=TEXT)

ax.set_title("🗺️  REGIONAL UNEMPLOYMENT HEATMAP — All States × All Months",
             color=WHITE, fontsize=15, fontweight="bold", pad=14)
fig.patch.set_facecolor(BG)
plt.tight_layout()
plt.savefig("images/08_heatmap.png", dpi=150, bbox_inches="tight", facecolor=BG)
plt.show()
print("Saved → images/08_heatmap.png")


## 11. Policy Insights Summary

In [ ]:
fig, ax = plt.subplots(figsize=(20, 8), facecolor=BG)
ax.set_facecolor(BG)
ax.axis("off")
ax.set_title("💡  POLICY INSIGHTS & RECOMMENDATIONS",
             color=WHITE, fontsize=16, fontweight="bold", pad=16, loc="left")

cards = [
    ("🦠 COVID-19\nShock",
     f"+{covid_avg-pre_covid:.1f}pp surge (87%)",
     f"Avg jumped {pre_covid:.1f}% → {covid_avg:.1f}%\nduring lockdown. Emergency\nfiscal stimulus is critical."),
    ("🔴 High-Risk\nStates",
     "Tripura · Haryana\nJharkhand · Bihar",
     "Chronic >18% unemployment.\nSpecial Economic Zones &\nskill centres urgently needed."),
    ("🌾 Rural-Urban\nGap",
     f"Urban {urban_avg:.1f}% vs Rural {rural_avg:.1f}%",
     "Urban informal workers lack\nsafety nets. Portable benefit\nschemes are essential."),
    ("👥 Labour Force\nDropout",
     "LPR fell in lockdown",
     "Discouraged worker effect.\nMGNREGA expansion &\nALMPs can re-engage them."),
    ("📅 Seasonal\nPattern",
     "Apr–May spike yearly",
     "Pre-monsoon job losses\nneed agri-credit & rural\npublic works programmes."),
    ("📈 Recovery\nInsight",
     "Jun 2020 bounce-back",
     "Unlock 1.0 showed rapid\nrecovery. Staged reopening\nprotects livelihoods."),
]

cw, ch = 0.148, 0.80
CARD2 = "#1c2035"
for i, (title, metric, body) in enumerate(cards):
    x = 0.008 + i*(cw+0.012)
    patch = mpatches.FancyBboxPatch(
        (x, 0.05), cw, ch, boxstyle="round,pad=0.018",
        linewidth=1.5, edgecolor=TEAL, facecolor=CARD2,
        transform=ax.transAxes, clip_on=False)
    ax.add_patch(patch)
    ax.text(x+cw/2, 0.84, title, ha="center", va="top",
            fontsize=10, fontweight="bold", color=TEAL, transform=ax.transAxes)
    ax.text(x+cw/2, 0.63, metric, ha="center", va="top",
            fontsize=9.5, fontweight="bold", color=YELLOW, transform=ax.transAxes)
    ax.text(x+cw/2, 0.44, body, ha="center", va="top",
            fontsize=8.5, color=TEXT, transform=ax.transAxes,
            multialignment="center", linespacing=1.6)

plt.savefig("images/09_policy_insights.png", dpi=150, bbox_inches="tight", facecolor=BG)
plt.show()
print("✅ All analysis complete! All charts saved to images/")


---

## ✅ Summary

| Analysis | Key Finding |
|----------|-------------|
| 🇮🇳 Overall | National avg unemployment: **11.8%** over the period |
| 🦠 COVID-19 | Avg surged from **9.5% → 17.8%** (+8.3pp, 87% jump) |
| 🔴 Worst State | Tripura: **28.4%** chronic average |
| 🟢 Best State | Meghalaya: **4.8%** average |
| 🌾 Rural/Urban | Urban (**13.2%**) > Rural (**10.3%**) by 2.9pp |
| 📅 Seasonal | April–May peak every year (lockdown + pre-monsoon) |
| 🔗 Correlation | Labour participation negatively correlated with unemployment (r ≈ -0.3) |

> 📌 **Run `streamlit run Streamlit_UI.py` for the full interactive dashboard**
